# Protein Surprisal Atlas — FULL PROTEOME Run (GPU / Colab)

Scores **all reviewed canonical human proteins** (~20k) with the validated sampled-mask estimator,
using the partitioned/resumable `score_proteome.py` on `main`.

**Why this notebook is different from the pilot one:** a full run takes a few hours, and Colab can
disconnect. So it writes all outputs (the QC table and the scored chunks) to **Google Drive**, and
`score_proteome.py` resumes from whatever chunks already exist. If the runtime dies, just re-run the
scoring cell — it skips everything already done.

**Before running:** Runtime → Change runtime type → **GPU**. Requires `score_proteome.py` merged on
`main` (PR #10).

In [ ]:
# --- GPU check ---
import torch
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
if not torch.cuda.is_available():
    print("WARNING: set Runtime -> Change runtime type -> GPU, then rerun.")

## 1. Mount Google Drive (persistent store)
All outputs live under `MyDrive/psa_proteome/` so they survive disconnects/runtime resets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PERSIST = "/content/drive/MyDrive/psa_proteome"
os.makedirs(PERSIST + "/data", exist_ok=True)
os.makedirs(PERSIST + "/results", exist_ok=True)
print("Persistent store:", PERSIST)

## 2. Clone repo + install deps (Colab's CUDA torch left untouched)

In [ ]:
REPO="/content/protein-surprisal-atlas"
if os.path.isdir(REPO):
    !cd {REPO} && git fetch --quiet origin main && git checkout --quiet main && git pull --quiet
else:
    !git clone --branch main https://github.com/glenritschel/protein-surprisal-atlas.git {REPO}
%cd {REPO}
!pip install -q transformers pyarrow pandas pyyaml tqdm biopython requests
!git log --oneline -1

## 3. Redirect data/ and results/ onto Drive
Symlink the repo's `data` and `results` to the persistent store so the QC table and all scored
chunks are written to Drive and survive a runtime reset.

In [ ]:
import subprocess
# replace repo's (gitignored) data/ and results/ with symlinks into Drive
for name in ["data", "results"]:
    tgt = f"{PERSIST}/{name}"
    !rm -rf {REPO}/{name}
    !ln -s {tgt} {REPO}/{name}
os.makedirs(f"{PERSIST}/data/interim", exist_ok=True)
os.makedirs(f"{PERSIST}/data/raw", exist_ok=True)
os.makedirs(f"{PERSIST}/results/tables", exist_ok=True)
os.makedirs(f"{PERSIST}/results/logs", exist_ok=True)
!ls -la {REPO}/data {REPO}/results
os.environ["PYTHONPATH"] = REPO
print("PYTHONPATH =", os.environ["PYTHONPATH"])

## 4. Config (full-proteome, sampled-mask, GPU)

In [ ]:
import yaml
with open("config.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["model"]["device"] = "auto"                 # picks cuda
cfg["model"]["scoring_method"] = "sampled_mask"
cfg["scoring"] = cfg.get("scoring", {})
cfg["scoring"]["sampled_mask_passes"] = 7
with open("colab_config.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print(open("colab_config.yaml").read())

## 5. Download + QC the full reviewed human proteome
~20k entries, a few MB. Skipped automatically if the QC table already exists on Drive.

In [ ]:
qc = f"{REPO}/data/interim/human_proteome_qc.parquet"
if os.path.exists(qc):
    import pandas as pd
    print("QC table already on Drive:", len(pd.read_parquet(qc)), "proteins — skipping download.")
else:
    !PYTHONPATH=$PWD python scripts/download_data.py --config colab_config.yaml
    import json
    print(json.dumps(json.load(open("data/interim/qc_report.json")), indent=2))

## 6. Smoke test on 50 proteins
Confirms scoring + the partitioned/chunked output before committing to the full run.

In [ ]:
!PYTHONPATH=$PWD python scripts/score_proteome.py --config colab_config.yaml --method sampled_mask --limit 50
import glob, pandas as pd
parts = sorted(glob.glob(f"{REPO}/results/tables/proteome_protein_scores_sampled_mask/part-*.parquet"))
print("chunk files:", len(parts))
if parts:
    df = pd.concat([pd.read_parquet(p) for p in parts])
    print("proteins scored:", df['uniprot_id'].nunique())
    display(df[['uniprot_id','gene_symbol','sequence_length','bits_per_residue']].head())

## 7. FULL RUN — score the whole proteome
Long (a few hours on a T4). **Resumable:** if Colab disconnects, just re-run this cell — it reads the
chunks already on Drive and skips completed proteins. Safe to run repeatedly.

In [ ]:
!PYTHONPATH=$PWD python scripts/score_proteome.py --config colab_config.yaml --method sampled_mask

## 8. Progress / verify

In [ ]:
import glob, pandas as pd, json, os
pdir = f"{REPO}/results/tables/proteome_protein_scores_sampled_mask"
parts = sorted(glob.glob(pdir + "/part-*.parquet"))
total = pd.read_parquet(f"{REPO}/data/interim/human_proteome_qc.parquet")['uniprot_id'].nunique()
done = pd.concat([pd.read_parquet(p, columns=['uniprot_id']) for p in parts])['uniprot_id'].nunique() if parts else 0
print(f"Scored {done} / {total} proteins across {len(parts)} chunk files.")
errf = f"{REPO}/results/logs/scoring_errors_proteome_sampled_mask.jsonl"
if os.path.exists(errf):
    n_err = sum(1 for _ in open(errf))
    print(f"Errors logged: {n_err} (see {errf})")
print("\nWhen done == total, the proteome is fully scored. Outputs persist on Drive at:")
print(pdir)

## 9. (Optional) Zip a copy of the scored tables to download
The results already persist on Drive. This just makes a local download if you want one.

In [ ]:
import shutil
shutil.make_archive("/content/proteome_scores", "zip",
                    f"{REPO}/results/tables")
from google.colab import files
print("Zipped:", os.path.getsize("/content/proteome_scores.zip")//(1024*1024), "MB")
# files.download("/content/proteome_scores.zip")   # uncomment if you want it locally